# Spectral Methods
## Chebyshev Collocation for Differential Equations — Exponential Convergence from Scratch

---

**Author:** Computational Mathematics Notebook Series  
**Topic:** Chebyshev Polynomials, Spectral Differentiation, Collocation Methods  
**Prerequisites:** Linear algebra, ODEs/PDEs, polynomial interpolation, Python/NumPy  
**Primary Reference:** Trefethen, L. N. (2000). *Spectral Methods in MATLAB*. SIAM.

## 1. Problem Statement & Motivation

### What Are Spectral Methods?

Spectral methods solve differential equations by approximating the solution as a finite sum of **global basis functions** — typically Fourier modes (for periodic problems) or Chebyshev/Legendre polynomials (for non-periodic problems on bounded intervals). Unlike finite differences or finite elements, which are *local* methods, spectral methods use information from the entire domain simultaneously.

For a function $u(x)$ on $[-1,1]$, a degree-$N$ Chebyshev spectral approximation is:

$$\boxed{u_N(x) = \sum_{k=0}^{N} \hat{u}_k \, T_k(x)}$$

where $T_k$ are the Chebyshev polynomials and $\hat{u}_k$ are the **spectral coefficients**.

### Why Exponential Convergence?

The key theorem: if $u$ is **analytic** on $[-1,1]$ (i.e., it extends analytically to a region of the complex plane), then the Chebyshev coefficients decay **geometrically**:

$$|\hat{u}_k| = O(\rho^{-k}) \quad \text{as} \quad k \to \infty$$

for some $\rho > 1$ determined by the size of the analytic region. This implies the approximation error satisfies:

$$\|u - u_N\|_\infty = O(\rho^{-N})$$

This is **spectral accuracy** — the error decreases exponentially in $N$, far faster than the algebraic $O(h^p)$ convergence of finite differences.

### Spectral vs. Finite Differences

| Property | Finite Differences | Spectral Methods |
|---|---|---|
| Basis functions | Local (stencil) | Global (polynomials/trig) |
| Convergence rate | Algebraic $O(h^p)$ | Exponential $O(\rho^{-N})$ |
| Matrix structure | Sparse, banded | Dense |
| Grid points needed | Many (100s–1000s) | Few (10s–100s) |
| Best for | Complex geometry | Smooth solutions |
| Implementation | Simple | Moderate |

For **smooth solutions** on simple domains, spectral methods are dramatically more efficient: the same accuracy requires orders of magnitude fewer degrees of freedom.

In [ ]:
%matplotlib inline

import numpy as np
from scipy import linalg as la
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

print("All imports successful.")

In [ ]:
# =============================================================================
# Constants and Configuration
# =============================================================================

SEED = 42
np.random.seed(SEED)

# Color palette for consistent plots
COLORS = {
    'primary':   '#2196F3',   # blue
    'secondary': '#F44336',   # red
    'accent':    '#4CAF50',   # green
    'highlight': '#FF9800',   # orange
    'purple':    '#9C27B0',   # purple
    'gray':      '#607D8B',   # gray
}

# Numerical tolerances
ATOL = 1e-10   # absolute tolerance for verifications
RTOL = 1e-8    # relative tolerance for convergence checks

print(f"Seed set to {SEED}")
print(f"Absolute tolerance: {ATOL}, Relative tolerance: {RTOL}")

## 2. Chebyshev Polynomials

### Definition and Properties

The **Chebyshev polynomials of the first kind** are defined on $[-1,1]$ by:

$$\boxed{T_n(x) = \cos(n \arccos x), \quad x \in [-1,1]}$$

This trigonometric definition immediately gives the recurrence relation (from the cosine addition formula $\cos((n+1)\theta) + \cos((n-1)\theta) = 2\cos\theta\cos(n\theta)$ with $x = \cos\theta$):

$$T_0(x) = 1, \quad T_1(x) = x, \quad T_{n+1}(x) = 2x\,T_n(x) - T_{n-1}(x)$$

### Key Properties

**Orthogonality** (with weight $w(x) = (1-x^2)^{-1/2}$):
$$\int_{-1}^{1} T_m(x)\,T_n(x)\,\frac{dx}{\sqrt{1-x^2}} = \begin{cases} 0 & m \neq n \\ \pi/2 & m = n \neq 0 \\ \pi & m = n = 0 \end{cases}$$

**Minimax property**: Among all monic polynomials of degree $n$, $2^{1-n}T_n(x)$ has the smallest maximum deviation from zero on $[-1,1]$:
$$\max_{x \in [-1,1]} |2^{1-n}T_n(x)| = 2^{1-n}$$
This is the **equioscillation theorem** — $T_n$ equioscillates $n+1$ times, achieving its maximum modulus of $1$ at $n+1$ points.

**Extrema and roots**: The $n+1$ extrema of $T_n$ occur at $x_j = \cos(j\pi/n)$, $j=0,\ldots,n$ — these are the **Gauss-Lobatto points**.

In [ ]:
# =============================================================================
# Chebyshev Polynomials — Definition and Visualization
# =============================================================================

def chebyshev_poly(n, x):
    """Evaluate Chebyshev polynomial T_n at points x using recurrence.

    Args:
        n (int): Degree of the Chebyshev polynomial (n >= 0).
        x (np.ndarray): Points at which to evaluate, shape (M,).

    Returns:
        np.ndarray: Values T_n(x), shape (M,).
    """
    x = np.asarray(x, dtype=float)
    if n == 0:
        return np.ones_like(x)
    if n == 1:
        return x.copy()
    T_prev = np.ones_like(x)   # T_{k-1}
    T_curr = x.copy()          # T_k
    for _ in range(2, n + 1):
        T_next = 2.0 * x * T_curr - T_prev
        T_prev = T_curr
        T_curr = T_next
    return T_curr


def chebyshev_poly_trig(n, x):
    """Evaluate T_n via the trigonometric definition T_n(x) = cos(n*arccos(x)).

    Args:
        n (int): Degree.
        x (np.ndarray): Points in [-1, 1], shape (M,).

    Returns:
        np.ndarray: Values T_n(x), shape (M,).
    """
    return np.cos(n * np.arccos(np.clip(x, -1.0, 1.0)))


# Verify both definitions agree
x_test = np.linspace(-1, 1, 200)
for n in range(7):
    err = np.max(np.abs(chebyshev_poly(n, x_test) - chebyshev_poly_trig(n, x_test)))
    assert err < ATOL, f"Mismatch at degree {n}: {err}"
print("Recurrence vs. trig definition agree for n=0..6 [PASS]")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_plot = np.linspace(-1, 1, 500)
cmap = plt.cm.viridis
degrees = range(6)

ax = axes[0]
for n in degrees:
    Tn = chebyshev_poly(n, x_plot)
    ax.plot(x_plot, Tn, color=cmap(n / 5), label=f'$T_{n}$')
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_title('Chebyshev Polynomials $T_0$ through $T_5$')
ax.set_xlabel('$x$')
ax.set_ylabel('$T_n(x)$')
ax.set_ylim(-1.3, 1.3)
ax.legend(ncol=2, fontsize=10)

ax = axes[1]
for n in degrees:
    theta = np.linspace(0, np.pi, 300)
    ax.plot(theta, np.cos(n * theta), color=cmap(n / 5), label=f'$T_{n}(\\cos\\theta)$')
ax.set_title('Chebyshev Polynomials as Cosines: $T_n(\\cos\\theta) = \\cos(n\\theta)$')
ax.set_xlabel('$\\theta$')
ax.set_ylabel('$\\cos(n\\theta)$')
ax.set_xticks([0, np.pi/2, np.pi])
ax.set_xticklabels(['$0$', '$\\pi/2$', '$\\pi$'])
ax.legend(ncol=2, fontsize=10)

plt.tight_layout()
plt.suptitle('Chebyshev Polynomials', fontsize=14, y=1.02)
plt.show()

In [ ]:
# =============================================================================
# Orthogonality Verification
# =============================================================================

# Numerically verify orthogonality: integral T_m * T_n / sqrt(1-x^2) dx
# Use Gauss-Chebyshev quadrature: sum at x_k = cos((2k-1)pi/(2M)), w_k = pi/M
M = 500
k = np.arange(1, M + 1)
xq = np.cos((2*k - 1) * np.pi / (2*M))   # Gauss-Chebyshev nodes
# weight pi/M for each node (uniform in theta)

print("Orthogonality check (should match pi/2 for m=n>0, pi for m=n=0, 0 for m!=n):")
for m, n in [(0,0), (1,1), (3,3), (2,3), (1,4)]:
    integral = (np.pi / M) * np.sum(chebyshev_poly(m, xq) * chebyshev_poly(n, xq))
    expected = np.pi if (m == n == 0) else (np.pi/2 if m == n else 0.0)
    status = 'PASS' if abs(integral - expected) < 1e-6 else 'FAIL'
    print(f"  <T_{m}, T_{n}> = {integral:8.5f}  (expected {expected:.5f})  [{status}]")

## 3. Chebyshev Nodes — Gauss-Lobatto Points

### Why Not Equispaced Nodes? The Runge Phenomenon

Polynomial interpolation at **equispaced** nodes suffers from catastrophic oscillations near the boundary as $N \to \infty$ — this is the **Runge phenomenon**, named after Carl Runge's 1901 example with $f(x) = 1/(1 + 25x^2)$.

The Gauss-Lobatto points solve this by clustering nodes near the endpoints:

$$\boxed{x_j = \cos\!\left(\frac{j\pi}{N}\right), \quad j = 0, 1, \ldots, N}$$

These are the **extrema of $T_N$** — the $N+1$ points where $|T_N|$ achieves its maximum of $1$. Key features:

- Symmetric: $x_{N-j} = -x_j$
- Endpoints included: $x_0 = 1$, $x_N = -1$ (essential for BCs in BVPs)
- Node density near $\pm 1$ is $O(N^2)$ times higher than the interior — this counteracts the endpoint amplification in polynomial interpolation
- The **Lebesgue constant** grows only as $O(\log N)$, compared to $O(2^N/N\log N)$ for equispaced nodes

In [ ]:
# =============================================================================
# Chebyshev Gauss-Lobatto Nodes — Visualization and Runge Phenomenon
# =============================================================================

def gauss_lobatto(N):
    """Compute N+1 Chebyshev-Gauss-Lobatto nodes on [-1, 1].

    Args:
        N (int): Number of intervals (N+1 nodes total).

    Returns:
        np.ndarray: Nodes x_j = cos(pi*j/N), shape (N+1,), ordered x_0=1 down to x_N=-1.
    """
    j = np.arange(N + 1)
    return np.cos(np.pi * j / N)


# --- Node distribution visualization ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Node positions for different N
ax = axes[0, 0]
for i, N in enumerate([8, 16, 32]):
    xgl = gauss_lobatto(N)
    y_offset = i * 0.3
    ax.plot(xgl, np.full_like(xgl, y_offset), 'o', markersize=4,
            color=list(COLORS.values())[i], label=f'N={N} ({N+1} nodes)')
ax.set_title('Gauss-Lobatto Node Distribution')
ax.set_xlabel('$x$')
ax.set_yticks([0, 0.3, 0.6])
ax.set_yticklabels(['N=8', 'N=16', 'N=32'])
ax.legend(fontsize=10)

# Node density: histogram
ax = axes[0, 1]
N_large = 100
xgl_large = gauss_lobatto(N_large)
xeq_large = np.linspace(-1, 1, N_large + 1)
ax.hist(xgl_large, bins=20, alpha=0.6, color=COLORS['primary'], label='Gauss-Lobatto', density=True)
ax.hist(xeq_large, bins=20, alpha=0.6, color=COLORS['secondary'], label='Equispaced', density=True)
# Theoretical density for Chebyshev: rho(x) = 1/(pi*sqrt(1-x^2))
x_th = np.linspace(-0.99, 0.99, 300)
ax.plot(x_th, 1.0 / (np.pi * np.sqrt(1 - x_th**2)), 'k--', lw=1.5, label='Theoretical $\\rho(x)$')
ax.set_title('Node Density: Chebyshev vs. Equispaced')
ax.set_xlabel('$x$')
ax.set_ylabel('Density')
ax.legend(fontsize=10)

# --- Runge phenomenon ---
runge = lambda x: 1.0 / (1.0 + 25.0 * x**2)
x_fine = np.linspace(-1, 1, 1000)
y_exact = runge(x_fine)

N_runge = 16

# Equispaced interpolation via numpy polyfit
x_eq = np.linspace(-1, 1, N_runge + 1)
y_eq = runge(x_eq)
p_eq = np.polyfit(x_eq, y_eq, N_runge)
y_eq_interp = np.polyval(p_eq, x_fine)

# Chebyshev-Lobatto interpolation via barycentric formula
def bary_interp(xnodes, ynodes, xeval):
    """Barycentric Lagrange interpolation.

    Args:
        xnodes (np.ndarray): Interpolation nodes, shape (n+1,).
        ynodes (np.ndarray): Function values at nodes, shape (n+1,).
        xeval  (np.ndarray): Evaluation points, shape (M,).

    Returns:
        np.ndarray: Interpolated values, shape (M,).
    """
    n = len(xnodes)
    # Barycentric weights: w_j = product_{k!=j}(x_j - x_k)^{-1}
    w = np.ones(n)
    for j in range(n):
        for k in range(n):
            if k != j:
                w[j] /= (xnodes[j] - xnodes[k])
    yeval = np.zeros(len(xeval))
    for i, xi in enumerate(xeval):
        diff = xi - xnodes
        exact_match = np.where(np.abs(diff) < 1e-14)[0]
        if len(exact_match) > 0:
            yeval[i] = ynodes[exact_match[0]]
        else:
            wdiff = w / diff
            yeval[i] = np.dot(wdiff, ynodes) / np.sum(wdiff)
    return yeval

x_cheb = gauss_lobatto(N_runge)
y_cheb_nodes = runge(x_cheb)
y_cheb_interp = bary_interp(x_cheb, y_cheb_nodes, x_fine)

ax = axes[1, 0]
ax.plot(x_fine, y_exact, 'k-', lw=2, label='Exact $f(x)$', zorder=5)
ax.plot(x_fine, y_eq_interp, color=COLORS['secondary'], ls='--', label=f'Equispaced N={N_runge}')
ax.plot(x_fine, y_cheb_interp, color=COLORS['primary'], ls='-', label=f'Chebyshev N={N_runge}')
ax.plot(x_eq, y_eq, 'rs', markersize=5, zorder=6)
ax.plot(x_cheb, y_cheb_nodes, 'b^', markersize=5, zorder=6)
ax.set_ylim(-0.5, 1.3)
ax.set_title('Runge Phenomenon: $f(x) = 1/(1+25x^2)$')
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.legend(fontsize=10)

# Error comparison
ax = axes[1, 1]
Nvals = range(4, 25)
err_eq_list, err_cheb_list = [], []
for N in Nvals:
    xeq_ = np.linspace(-1, 1, N + 1)
    p_ = np.polyfit(xeq_, runge(xeq_), N)
    err_eq_list.append(np.max(np.abs(np.polyval(p_, x_fine) - y_exact)))

    xch_ = gauss_lobatto(N)
    ych_ = bary_interp(xch_, runge(xch_), x_fine)
    err_cheb_list.append(np.max(np.abs(ych_ - y_exact)))

ax.semilogy(list(Nvals), err_eq_list, 'rs--', label='Equispaced', markersize=5)
ax.semilogy(list(Nvals), err_cheb_list, 'b^-', label='Chebyshev (GL)', markersize=5)
ax.set_title('Max Interpolation Error vs. $N$')
ax.set_xlabel('$N$ (degree)')
ax.set_ylabel('$\\|error\\|_\\infty$')
ax.legend(fontsize=10)

plt.suptitle('Chebyshev Nodes and Runge Phenomenon', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Chebyshev Differentiation Matrix

### Derivation

Given the Gauss-Lobatto nodes $x_0 > x_1 > \cdots > x_N$ with $x_j = \cos(j\pi/N)$, we want a matrix $D \in \mathbb{R}^{(N+1)\times(N+1)}$ such that:

$$(Du)_j \approx u'(x_j)$$

The idea: interpolate $u$ by the unique polynomial $p_N$ of degree $\leq N$ through the $N+1$ nodes, then differentiate $p_N$ exactly.

Using the **Lagrange barycentric formula**, the interpolating polynomial is:
$$p_N(x) = \sum_{k=0}^{N} u_k \ell_k(x), \quad \ell_k(x) = \prod_{j \neq k} \frac{x - x_j}{x_k - x_j}$$

Differentiating: $p_N'(x_j) = \sum_{k=0}^{N} D_{jk} u_k$, where the entries are:

$$\boxed{D_{jk} = \begin{cases}
\dfrac{c_j}{c_k} \dfrac{(-1)^{j+k}}{x_j - x_k} & j \neq k \\[6pt]
-\displaystyle\sum_{k \neq j} D_{jk} & j = k
\end{cases}}$$

where $c_0 = c_N = 2$ and $c_j = 1$ for $0 < j < N$. The diagonal entries follow from the **negative sum trick** (ensuring $D\mathbf{1} = \mathbf{0}$ since the derivative of a constant is zero).

The full $D$ matrix can be written compactly:

- Corner entries: $D_{00} = (2N^2+1)/6$, $D_{NN} = -(2N^2+1)/6$
- Off-diagonal: $D_{jk} = \frac{c_j}{c_k} \frac{(-1)^{j+k}}{x_j - x_k}$ for $j \neq k$
- Diagonal: $D_{jj} = -\frac{x_j}{2(1-x_j^2)}$ for $0 < j < N$ (from negative sum trick)

In [ ]:
# =============================================================================
# cheb(N): Chebyshev Differentiation Matrix from Scratch
# =============================================================================

def cheb(N):
    """Compute the Chebyshev spectral differentiation matrix of size (N+1)x(N+1).

    Uses the barycentric Lagrange formula on Gauss-Lobatto nodes
    x_j = cos(pi*j/N), j = 0, ..., N.

    Args:
        N (int): Polynomial degree. Must be >= 1.

    Returns:
        tuple:
            D (np.ndarray): Differentiation matrix, shape (N+1, N+1).
                D[j,k] approximates d/dx evaluated at x_j applied to the
                k-th Lagrange basis function. So (D @ u)[j] ~ u'(x_j).
            x (np.ndarray): Gauss-Lobatto nodes, shape (N+1,).
                x[0] = 1, x[N] = -1.
    """
    if N == 0:
        return np.array([[0.0]]), np.array([1.0])

    # Gauss-Lobatto nodes: x_j = cos(pi*j/N), j=0,...,N
    j = np.arange(N + 1)
    x = np.cos(np.pi * j / N)    # shape (N+1,)

    # Barycentric weights c_j (half-weight for endpoints)
    c = np.ones(N + 1)
    c[0] = 2.0
    c[N] = 2.0
    c *= (-1.0) ** j              # sign alternation

    # Off-diagonal entries: D[i,j] = (c[i]/c[j]) / (x[i] - x[j])
    X = np.tile(x, (N + 1, 1))   # shape (N+1, N+1), X[i,j] = x[j]
    dX = X - X.T                  # dX[i,j] = x[i] - x[j]  (0 on diagonal)

    # Avoid division by zero on diagonal
    with np.errstate(divide='ignore', invalid='ignore'):
        D = np.outer(c, 1.0 / c) / np.where(dX == 0, 1.0, dX)

    # Diagonal: negative row sum (negative sum trick: D*1 = 0)
    np.fill_diagonal(D, 0.0)
    np.fill_diagonal(D, -D.sum(axis=1))

    return D, x


print("cheb() function defined.")
print()

# Quick sanity: D for N=4
D4, x4 = cheb(4)
print(f"N=4 nodes: {x4.round(4)}")
print(f"N=4 D matrix (rounded):")
print(np.round(D4, 3))

In [ ]:
# =============================================================================
# Verification: Known Derivatives
# =============================================================================

N_verify = 20
D, x = cheb(N_verify)

print("=" * 60)
print(f"Verification of cheb({N_verify}) differentiation matrix")
print("=" * 60)

# Test 1: Derivative of sin(x)
u = np.sin(x)
Du_spectral = D @ u
Du_exact = np.cos(x)
err1 = np.max(np.abs(Du_spectral - Du_exact))
print(f"\nd/dx [sin(x)]:  max error = {err1:.2e}  [{'PASS' if err1 < 1e-10 else 'FAIL'}]")

# Test 2: Derivative of exp(x)
u = np.exp(x)
Du_spectral = D @ u
Du_exact = np.exp(x)
err2 = np.max(np.abs(Du_spectral - Du_exact))
print(f"d/dx [exp(x)]:  max error = {err2:.2e}  [{'PASS' if err2 < 1e-10 else 'FAIL'}]")

# Test 3: Derivative of x^5
u = x**5
Du_spectral = D @ u
Du_exact = 5.0 * x**4
err3 = np.max(np.abs(Du_spectral - Du_exact))
print(f"d/dx [x^5]:     max error = {err3:.2e}  [{'PASS' if err3 < 1e-10 else 'FAIL'}]")

# Test 4: Second derivative via D^2
u = np.sin(3.0 * x)
D2u_spectral = D @ D @ u
D2u_exact = -9.0 * np.sin(3.0 * x)
err4 = np.max(np.abs(D2u_spectral - D2u_exact))
print(f"d^2/dx^2 [sin(3x)]: max error = {err4:.2e}  [{'PASS' if err4 < 1e-10 else 'FAIL'}]")

# Test 5: Negative sum trick (D*1 = 0)
ones = np.ones(N_verify + 1)
Dones_err = np.max(np.abs(D @ ones))
print(f"Negative sum (D*1=0): max |D*1| = {Dones_err:.2e}  [{'PASS' if Dones_err < 1e-12 else 'FAIL'}]")

print()
print("All verifications complete.")

In [ ]:
# =============================================================================
# Visualize Differentiation Matrix Structure
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

D16, x16 = cheb(16)

# Matrix heatmap
im = axes[0].imshow(D16, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Chebyshev $D_{16}$ matrix')
axes[0].set_xlabel('column $k$')
axes[0].set_ylabel('row $j$')

# Off-diagonal magnitude
D_abs = np.abs(D16)
np.fill_diagonal(D_abs, 0)  # hide diagonal
im2 = axes[1].imshow(np.log10(D_abs + 1e-15), cmap='viridis', aspect='auto')
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('$\\log_{10}|D_{jk}|$ (off-diagonal)')
axes[1].set_xlabel('column $k$')
axes[1].set_ylabel('row $j$')

# Spectral derivative of sin(5x) + cos(3x)
f = lambda x: np.sin(5*x) + np.cos(3*x)
df = lambda x: 5*np.cos(5*x) - 3*np.sin(3*x)
u_nodes = f(x16)
du_spectral = D16 @ u_nodes
x_fine = np.linspace(-1, 1, 500)
axes[2].plot(x_fine, df(x_fine), 'k-', lw=2, label="Exact $u'$", zorder=5)
axes[2].plot(x16, du_spectral, 'ro', markersize=6, label='Spectral $Du$', zorder=6)
axes[2].set_title("Spectral derivative of $\\sin(5x)+\\cos(3x)$")
axes[2].set_xlabel('$x$')
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. Spectral Collocation for Boundary Value Problems

### Setting Up the Collocation System

For a general second-order BVP on $[-1,1]$:
$$u''(x) + p(x)u'(x) + q(x)u(x) = f(x), \quad u(-1) = \alpha, \quad u(1) = \beta$$

We seek a polynomial $u_N$ that **collocates** (satisfies the equation exactly) at the interior Gauss-Lobatto nodes $x_1, \ldots, x_{N-1}$, while enforcing the BCs at $x_0 = 1$ and $x_N = -1$.

Let $D = D_N$ be the $(N+1)\times(N+1)$ Chebyshev differentiation matrix, and $\mathbf{u} = [u_0, u_1, \ldots, u_N]^T$ the unknown values at nodes. Then:
- $D^2 \mathbf{u}$ approximates $\mathbf{u}''$ at each node
- $D \mathbf{u}$ approximates $\mathbf{u}'$ at each node

The interior collocation equations $(j = 1, \ldots, N-1)$ give:
$$(D^2)_{j,:}\mathbf{u} + p(x_j)(D)_{j,:}\mathbf{u} + q(x_j)u_j = f(x_j)$$

The full system: **replace rows 0 and $N$** with the boundary conditions:

$$\boxed{\begin{pmatrix} \mathbf{e}_0^T \\ (D^2 + P D + Q)_{1:N-1,:} \\ \mathbf{e}_N^T \end{pmatrix} \mathbf{u} = \begin{pmatrix} \beta \\ \mathbf{f}_{1:N-1} \\ \alpha \end{pmatrix}}$$

where $P = \text{diag}(p(x_j))$, $Q = \text{diag}(q(x_j))$, $\mathbf{e}_0, \mathbf{e}_N$ are unit vectors, and $x_0=1, x_N=-1$.

In [ ]:
# =============================================================================
# BVP Solver: Spectral Collocation (General Variable-Coefficient)
# =============================================================================

def spectral_bvp(N, p_func, q_func, f_func, alpha, beta):
    """Solve u'' + p(x)*u' + q(x)*u = f(x) on [-1,1], u(-1)=alpha, u(1)=beta.

    Uses Chebyshev collocation at N+1 Gauss-Lobatto nodes.

    Args:
        N (int): Polynomial degree (N+1 nodes).
        p_func (callable): Coefficient p(x), shape (N+1,) -> (N+1,).
        q_func (callable): Coefficient q(x), shape (N+1,) -> (N+1,).
        f_func (callable): Right-hand side f(x), shape (N+1,) -> (N+1,).
        alpha (float): Dirichlet BC at x = -1 (u(-1) = alpha, node x_N).
        beta  (float): Dirichlet BC at x = +1 (u(+1) = beta,  node x_0).

    Returns:
        tuple:
            u (np.ndarray): Solution values at Gauss-Lobatto nodes, shape (N+1,).
            x (np.ndarray): Gauss-Lobatto nodes, shape (N+1,).
    """
    D, x = cheb(N)
    D2 = D @ D

    p = p_func(x)
    q = q_func(x)
    f = f_func(x)

    # Build the (N+1)x(N+1) system matrix A
    A = D2 + np.diag(p) @ D + np.diag(q)
    rhs = f.copy()

    # Enforce BCs: row 0 -> u(x_0) = u(+1) = beta
    A[0, :]  = 0.0; A[0, 0]  = 1.0; rhs[0]  = beta
    # row N -> u(x_N) = u(-1) = alpha
    A[-1, :] = 0.0; A[-1, -1] = 1.0; rhs[-1] = alpha

    u = la.solve(A, rhs)
    return u, x


print("spectral_bvp() defined.")

In [ ]:
# =============================================================================
# BVP Example 1: u'' = e^{4x}, u(-1) = u(1) = 0
# Exact: u(x) = (e^{4x} - x*sinh(4) - cosh(4)) / 16
# =============================================================================

# Exact solution derivation:
# u'' = e^{4x}  =>  u = e^{4x}/16 + C1*x + C2
# u(1) = 0:  e^4/16 + C1 + C2 = 0
# u(-1) = 0: e^{-4}/16 - C1 + C2 = 0
# => C1 = (e^4 - e^{-4})/32 = sinh(4)/16
# => C2 = -(e^4 + e^{-4})/32 = -cosh(4)/16
u_exact1 = lambda x: (np.exp(4*x) - x*np.sinh(4) - np.cosh(4)) / 16.0

N_test = 16
u_spec, x_spec = spectral_bvp(
    N_test,
    p_func=lambda x: np.zeros_like(x),
    q_func=lambda x: np.zeros_like(x),
    f_func=lambda x: np.exp(4*x),
    alpha=0.0, beta=0.0
)

err = np.max(np.abs(u_spec - u_exact1(x_spec)))
print(f"BVP 1 (u'' = e^4x): max error with N={N_test}: {err:.2e}  [{'PASS' if err < 1e-10 else 'FAIL'}]")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x_fine = np.linspace(-1, 1, 500)

ax = axes[0]
ax.plot(x_fine, u_exact1(x_fine), 'k-', lw=2, label='Exact solution', zorder=5)
ax.plot(x_spec, u_spec, 'bo', markersize=6, label=f'Spectral N={N_test}', zorder=6)
ax.set_title("BVP 1: $u'' = e^{4x}$, $u(\\pm 1)=0$")
ax.set_xlabel('$x$'); ax.set_ylabel('$u(x)$')
ax.legend(fontsize=10)

ax = axes[1]
pointwise_err = np.abs(u_spec - u_exact1(x_spec))
ax.semilogy(x_spec, pointwise_err + 1e-18, 'bs-', markersize=4, label='Pointwise error')
ax.set_title('Pointwise Error at Collocation Nodes')
ax.set_xlabel('$x$'); ax.set_ylabel('$|u_{\\rm spec} - u_{\\rm exact}|$')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# BVP Example 2: u'' + sin(x)*u' - cos(x)*u = x^2, u(-1) = 1, u(1) = 0
# Solved spectrally (no closed-form exact — verify via residual and refinement)
# =============================================================================

# We use two different N values and check convergence
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

solutions = {}
for N in [12, 24, 48]:
    u_sol, x_sol = spectral_bvp(
        N,
        p_func=lambda x: np.sin(x),
        q_func=lambda x: -np.cos(x),
        f_func=lambda x: x**2,
        alpha=1.0, beta=0.0
    )
    solutions[N] = (x_sol, u_sol)

# Reference solution at N=48
x_ref, u_ref = solutions[48]

ax = axes[0]
for N, (xs, us) in solutions.items():
    ax.plot(xs, us, 'o-', markersize=3, label=f'N={N}')
ax.set_title("BVP 2: $u''+\\sin(x)u'-\\cos(x)u = x^2$, $u(-1)=1$, $u(1)=0$")
ax.set_xlabel('$x$'); ax.set_ylabel('$u(x)$')
ax.legend(fontsize=10)

# Convergence: compare to N=48 reference
ax = axes[1]
Nvals_conv = [6, 8, 10, 12, 14, 16, 18, 20, 24, 32]
errors_ex2 = []
for N in Nvals_conv:
    u_sol, x_sol = spectral_bvp(
        N,
        p_func=lambda x: np.sin(x),
        q_func=lambda x: -np.cos(x),
        f_func=lambda x: x**2,
        alpha=1.0, beta=0.0
    )
    # Interpolate reference to x_sol via bary_interp
    u_ref_at_sol = bary_interp(x_ref, u_ref, x_sol)
    errors_ex2.append(np.max(np.abs(u_sol - u_ref_at_sol)))

ax.semilogy(Nvals_conv, errors_ex2, 'gs-', markersize=6, label='Error vs N=48 ref')
ax.set_title('Convergence of BVP 2 (vs N=48 reference)')
ax.set_xlabel('$N$'); ax.set_ylabel('Max error')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"BVP 2 converges to machine precision around N~20.")

In [ ]:
# =============================================================================
# BVP Example 3: Airy-like equation  u'' - x*u = 0, u(-1)=1, u(1)=Ai(1)/Ai(-1)
# Exact solution: u(x) = Ai(x)/Ai(-1)   where Ai is the Airy function
# We use scipy.special only for the exact solution, NOT the BVP solver
# =============================================================================

from scipy.special import airy

Ai_neg1 = airy(-1.0)[0]   # Ai(-1)
Ai_pos1 = airy(1.0)[0]    # Ai(1)
bc_left  = 1.0             # u(-1) = 1
bc_right = Ai_pos1 / Ai_neg1   # u(1)

u_exact_airy = lambda x: airy(x)[0] / Ai_neg1

N_airy = 18
u_airy, x_airy = spectral_bvp(
    N_airy,
    p_func=lambda x: np.zeros_like(x),
    q_func=lambda x: -x,             # q(x) = -x  =>  u'' - x*u = 0
    f_func=lambda x: np.zeros_like(x),
    alpha=bc_left, beta=bc_right
)

err_airy = np.max(np.abs(u_airy - u_exact_airy(x_airy)))
print(f"BVP 3 (Airy): max error with N={N_airy}: {err_airy:.2e}  [{'PASS' if err_airy < 1e-10 else 'FAIL'}]")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x_fine = np.linspace(-1, 1, 500)

ax = axes[0]
ax.plot(x_fine, u_exact_airy(x_fine), 'k-', lw=2, label='Exact Airy $u$', zorder=5)
ax.plot(x_airy, u_airy, 'ro', markersize=6, label=f'Spectral N={N_airy}', zorder=6)
ax.set_title("BVP 3: $u'' - xu = 0$ (Airy equation)")
ax.set_xlabel('$x$'); ax.set_ylabel('$u(x)$')
ax.legend(fontsize=10)

ax = axes[1]
Nvals_airy = range(4, 30)
errs_airy = []
for N in Nvals_airy:
    u_s, x_s = spectral_bvp(
        N,
        p_func=lambda x: np.zeros_like(x),
        q_func=lambda x: -x,
        f_func=lambda x: np.zeros_like(x),
        alpha=bc_left, beta=bc_right
    )
    errs_airy.append(np.max(np.abs(u_s - u_exact_airy(x_s))))

ax.semilogy(list(Nvals_airy), errs_airy, 'r^-', markersize=5, label='Spectral error')
ax.set_title('Convergence: Airy BVP')
ax.set_xlabel('$N$'); ax.set_ylabel('$\\|u - u_{\\rm exact}\\|_\\infty$')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 6. Exponential Convergence Study

### Theoretical Rate

For an analytic function $u$ that extends holomorphically to the Bernstein ellipse $E_\rho$ (ellipse with foci $\pm 1$ and sum of semi-axes $= \rho > 1$), the best polynomial approximation error satisfies:

$$\|u - p_N^*\|_\infty \leq \frac{M}{\rho^N (\rho - 1)}$$

where $M = \max_{z \in E_\rho} |u(z)|$. Plotting $\log(\text{error})$ vs. $N$ should yield a **straight line** with slope $-\log(\rho)$.

For $u(x) = e^x$: the function is entire ($\rho = \infty$), so convergence is supergeometric. For $u(x) = 1/(1 + \alpha^2 x^2)$ with poles at $x = \pm i/\alpha$: $\rho = \alpha + \sqrt{\alpha^2+1}$.

In [ ]:
# =============================================================================
# Convergence Study: Exponential vs Algebraic Rate
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- BVP: u'' = sin(pi*x), exact: u = -sin(pi*x)/pi^2, u(pm1)=0 ---
# This is analytic, so we expect exponential convergence
u_ex_sin = lambda x: -np.sin(np.pi * x) / np.pi**2

Nvals = list(range(3, 32))
errs_sin = []
for N in Nvals:
    u_s, x_s = spectral_bvp(
        N,
        p_func=lambda x: np.zeros_like(x),
        q_func=lambda x: np.zeros_like(x),
        f_func=lambda x: np.sin(np.pi * x),
        alpha=0.0, beta=0.0
    )
    errs_sin.append(np.max(np.abs(u_s - u_ex_sin(x_s))))

ax = axes[0]
ax.semilogy(Nvals, errs_sin, 'b^-', markersize=5, label="$u''=\\sin(\\pi x)$ (spectral)")

# BVP: u'' = e^{4x}, exact known
errs_exp4 = []
for N in Nvals:
    u_s, x_s = spectral_bvp(
        N,
        p_func=lambda x: np.zeros_like(x),
        q_func=lambda x: np.zeros_like(x),
        f_func=lambda x: np.exp(4*x),
        alpha=0.0, beta=0.0
    )
    errs_exp4.append(np.max(np.abs(u_s - u_exact1(x_s))))

ax.semilogy(Nvals, errs_exp4, 'rs-', markersize=5, label="$u''=e^{4x}$ (spectral)")

# BVP: Airy
errs_airy_conv = []
for N in Nvals:
    u_s, x_s = spectral_bvp(
        N,
        p_func=lambda x: np.zeros_like(x),
        q_func=lambda x: -x,
        f_func=lambda x: np.zeros_like(x),
        alpha=bc_left, beta=bc_right
    )
    errs_airy_conv.append(np.max(np.abs(u_s - u_exact_airy(x_s))))

ax.semilogy(Nvals, errs_airy_conv, 'g^-', markersize=5, label='Airy (spectral)')

ax.axhline(1e-15, color='k', ls=':', lw=1, label='Machine $\\epsilon$')
ax.set_title('Spectral Convergence: Error vs. $N$ (all BVPs)')
ax.set_xlabel('$N$ (polynomial degree)')
ax.set_ylabel('$\\|u - u_{\\rm exact}\\|_\\infty$')
ax.legend(fontsize=10)

# --- Linear convergence plot to extract rate ---
ax = axes[1]
log_errs = np.log10(np.array(errs_exp4) + 1e-16)
# Fit a line to the linear part (before saturation at ~machine eps)
linear_mask = np.array(errs_exp4) > 1e-13
Nfit = np.array(Nvals)[linear_mask]
efit = log_errs[linear_mask]
if len(Nfit) > 2:
    slope, intercept = np.polyfit(Nfit, efit, 1)
    ax.semilogy(Nvals, errs_exp4, 'rs-', markersize=5, label='Error (spectral)')
    x_fit = np.array([Nvals[0], Nvals[-1]])
    ax.semilogy(x_fit, 10**(slope * x_fit + intercept), 'k--', lw=1.5,
                label=f'Fit: slope={slope:.3f} (per $N$)')
    ax.set_title(f'Exponential Convergence Rate\n'
                 f'slope $\\approx$ {slope:.3f} in $\\log_{{10}}$ → $10^{{{slope:.2f}}}$ per degree')

ax.set_xlabel('$N$')
ax.set_ylabel('$\\|u - u_{\\rm exact}\\|_\\infty$')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Spectral methods achieve machine precision in ~15-20 degrees for analytic solutions.")

## 7. Comparison with Finite Differences

### Finite Difference Discretization

For the same BVP $u'' = f(x)$ on $[-1,1]$ with $u(\pm 1)=0$, using $M$ interior equispaced points $x_i = -1 + i\,h$, $h = 2/(M+1)$, $i = 1, \ldots, M$, the 2nd-order centered FD approximation is:

$$\frac{u_{i-1} - 2u_i + u_{i+1}}{h^2} = f(x_i), \quad i = 1, \ldots, M$$

This gives a tridiagonal system with error $O(h^2) = O(M^{-2})$.

### Expected Behavior

- **Finite differences**: $\|e\|_\infty \propto M^{-2}$ — straight line with slope $-2$ on log-log plot
- **Spectral**: $\|e\|_\infty \propto \rho^{-N}$ — straight line with negative slope on log-linear (semi-log) plot

For the same accuracy, spectral methods may need 10–100× fewer degrees of freedom than FD.

In [ ]:
# =============================================================================
# Finite Difference BVP Solver (2nd-order centered)
# =============================================================================

def fd_bvp(M, f_func, alpha=0.0, beta=0.0):
    """Solve u'' = f(x) on [-1,1], u(-1)=alpha, u(1)=beta using 2nd-order FD.

    Args:
        M (int): Number of interior grid points.
        f_func (callable): RHS function f(x), (M,) -> (M,).
        alpha (float): BC at x = -1.
        beta  (float): BC at x = +1.

    Returns:
        tuple:
            u (np.ndarray): Solution at interior nodes, shape (M,).
            x_int (np.ndarray): Interior node positions, shape (M,).
    """
    h = 2.0 / (M + 1)
    x_int = -1.0 + np.arange(1, M + 1) * h   # interior nodes

    # Tridiagonal system: (1/h^2)*tridiag(1, -2, 1) * u = f
    diag_main = -2.0 * np.ones(M) / h**2
    diag_off  =  1.0 * np.ones(M - 1) / h**2
    A_fd = np.diag(diag_main) + np.diag(diag_off, 1) + np.diag(diag_off, -1)

    rhs = f_func(x_int).copy()
    # Adjust for BCs
    rhs[0]  -= alpha / h**2
    rhs[-1] -= beta  / h**2

    u = la.solve(A_fd, rhs)
    return u, x_int


# =============================================================================
# Side-by-side comparison: u'' = sin(pi*x), u(pm1)=0
# =============================================================================

u_exact_sin = lambda x: -np.sin(np.pi * x) / np.pi**2
f_sin = lambda x: np.sin(np.pi * x)

# Finite differences: M interior points
M_vals = [4, 8, 16, 32, 64, 128, 256, 512]
errs_fd = []
for M in M_vals:
    u_fd, x_fd = fd_bvp(M, f_sin, alpha=0.0, beta=0.0)
    errs_fd.append(np.max(np.abs(u_fd - u_exact_sin(x_fd))))

# Spectral: N polynomial degree (N+1 nodes)
N_vals_spec = list(range(2, 22))
errs_spec = []
for N in N_vals_spec:
    u_s, x_s = spectral_bvp(
        N, lambda x: np.zeros_like(x), lambda x: np.zeros_like(x),
        f_sin, 0.0, 0.0
    )
    errs_spec.append(np.max(np.abs(u_s - u_exact_sin(x_s))))


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error vs N (degrees of freedom)
ax = axes[0]
dof_fd = np.array(M_vals)           # interior FD points
dof_spec = np.array(N_vals_spec) + 1  # N+1 total spectral nodes

ax.loglog(dof_fd, errs_fd, 'rs-', markersize=6, label='Finite Differences (2nd order)')
# Algebraic reference: O(DOF^{-2})
ref_fd = errs_fd[0] * (dof_fd[0] / dof_fd)**2
ax.loglog(dof_fd, ref_fd, 'r--', lw=1, label='$O(M^{-2})$ reference')

ax.semilogy(dof_spec, errs_spec, 'b^-', markersize=6, label='Spectral (Chebyshev)')

ax.set_title('Error vs. Degrees of Freedom')
ax.set_xlabel('Degrees of freedom (DOF)')
ax.set_ylabel('$\\|u - u_{\\rm exact}\\|_\\infty$')
ax.legend(fontsize=10)
ax.set_xlim(left=3)

# Semi-log: error vs N (spectral)
ax = axes[1]
ax.semilogy(N_vals_spec, errs_spec, 'b^-', markersize=6, label='Spectral')
ax.semilogy(M_vals, errs_fd, 'rs-', markersize=6, label='Finite Differences (M)')
ax.axhline(1e-15, color='k', ls=':', lw=1, label='Machine $\\epsilon$')
ax.set_title('Error vs. $N$ (semi-log): Algebraic vs. Exponential')
ax.set_xlabel('$N$ or $M$ (problem size)')
ax.set_ylabel('$\\|error\\|_\\infty$')
ax.legend(fontsize=10)

plt.suptitle('Spectral vs. Finite Differences: $u\'\'=\\sin(\\pi x)$, $u(\\pm 1)=0$', fontsize=13)
plt.tight_layout()
plt.show()

# Summary
# Find N at which spectral reaches 1e-10
spec_arr = np.array(errs_spec)
N_target = N_vals_spec[np.argmax(spec_arr < 1e-10)] if np.any(spec_arr < 1e-10) else 'N/A'
# Find M at which FD reaches ~same accuracy as spectral at N_target
print(f"Spectral reaches < 1e-10 at N = {N_target} ({N_target+1} DOF)")
# Closest FD accuracy to 1e-10
fd_arr = np.array(errs_fd)
M_target = M_vals[np.argmin(np.abs(fd_arr - 1e-10))]
print(f"FD closest to 1e-10: M = {M_target} DOF, error = {fd_arr[M_vals.index(M_target)]:.2e}")
print(f"Spectral advantage: ~{M_target // (N_target+1)}x fewer DOF for comparable accuracy")

## 8. 2D Problems — Tensor Product Grids

### Poisson Equation on $[-1,1]^2$

Consider the 2D Poisson equation:
$$\nabla^2 u = u_{xx} + u_{yy} = f(x,y), \quad (x,y) \in [-1,1]^2$$
with Dirichlet BCs $u = g$ on the boundary.

### Kronecker Product Construction

Using $N+1$ Gauss-Lobatto nodes in each direction, let $D_x$ and $D_y$ be the $(N+1)\times(N+1)$ Chebyshev differentiation matrices in $x$ and $y$. The solution values are arranged in a matrix $U \in \mathbb{R}^{(N+1)\times(N+1)}$ where $U_{jk} = u(x_j, y_k)$, or vectorized as $\mathbf{u} = \text{vec}(U)$.

The Laplacian operator on the tensor product grid is:

$$\boxed{L = I_y \otimes D_x^2 + D_y^2 \otimes I_x}$$

where $\otimes$ denotes the Kronecker product. The system $L\,\mathbf{u} = \mathbf{f}$ is of size $(N+1)^2 \times (N+1)^2$, solved directly with `scipy.linalg.solve`.

Boundary conditions are enforced by replacing the rows corresponding to boundary nodes with identity rows.

In [ ]:
# =============================================================================
# 2D Poisson Solver via Kronecker Products
# =============================================================================

def spectral_poisson_2d(N, f_func, g_func):
    """Solve Poisson equation u_xx + u_yy = f on [-1,1]^2, u=g on boundary.

    Uses Chebyshev collocation with tensor product grid.

    Args:
        N (int): Polynomial degree (N+1 nodes per direction, (N+1)^2 total).
        f_func (callable): RHS f(X, Y) for meshgrid arrays X, Y of shape (N+1,N+1).
        g_func (callable): BC function g(X, Y), same shape.

    Returns:
        tuple:
            U (np.ndarray): Solution on grid, shape (N+1, N+1). U[i,j] = u(x_i, y_j).
            x (np.ndarray): Gauss-Lobatto nodes in x, shape (N+1,).
            y (np.ndarray): Gauss-Lobatto nodes in y, shape (N+1,).
    """
    D, x = cheb(N)
    D2 = D @ D
    y  = x.copy()  # same nodes in y (square domain)

    # Kronecker product Laplacian: L = I (x) D2 + D2 (x) I
    # Ordering: u[i*(N+1)+j] = U[i,j] where i=x-index, j=y-index
    I = np.eye(N + 1)
    L = np.kron(I, D2) + np.kron(D2, I)   # shape ((N+1)^2, (N+1)^2)

    # Build meshgrid: X[i,j] = x[i], Y[i,j] = y[j]
    X, Y = np.meshgrid(x, y, indexing='ij')   # shape (N+1, N+1)

    # RHS vector
    F = f_func(X, Y).ravel()   # shape ((N+1)^2,)
    G = g_func(X, Y).ravel()   # BC values

    # Identify boundary indices: nodes where i=0, i=N, j=0, or j=N
    M = N + 1
    is_boundary = np.zeros(M * M, dtype=bool)
    for idx in range(M * M):
        i, j = divmod(idx, M)
        if i == 0 or i == N or j == 0 or j == N:
            is_boundary[idx] = True

    # Replace BC rows with identity
    L[is_boundary, :]         = 0.0
    L[is_boundary, is_boundary] = 1.0  # This won't work directly; fix below
    # Correct way: set row to e_k
    bc_idx = np.where(is_boundary)[0]
    L[is_boundary, :] = 0.0
    for k in bc_idx:
        L[k, k] = 1.0

    rhs = F.copy()
    rhs[is_boundary] = G[is_boundary]

    u_vec = la.solve(L, rhs)
    U = u_vec.reshape(M, M)
    return U, x, y


print("spectral_poisson_2d() defined.")

In [ ]:
# =============================================================================
# Test 2D Poisson: Exact solution u(x,y) = sin(pi*x)*sin(pi*y)
# => f = -2*pi^2 * sin(pi*x)*sin(pi*y), BC: u=0 on boundary
# =============================================================================

u_exact_2d = lambda X, Y: np.sin(np.pi * X) * np.sin(np.pi * Y)
f_2d       = lambda X, Y: -2.0 * np.pi**2 * np.sin(np.pi * X) * np.sin(np.pi * Y)
g_2d       = lambda X, Y: np.zeros_like(X)   # homogeneous Dirichlet

N_2d = 20
U_spec, x_2d, y_2d = spectral_poisson_2d(N_2d, f_2d, g_2d)

X2, Y2 = np.meshgrid(x_2d, y_2d, indexing='ij')
U_exact = u_exact_2d(X2, Y2)
err_2d = np.max(np.abs(U_spec - U_exact))
print(f"2D Poisson (N={N_2d}): max error = {err_2d:.2e}  [{'PASS' if err_2d < 1e-8 else 'FAIL'}]")

# Visualization
fig = plt.figure(figsize=(16, 5))

# Exact solution
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(X2, Y2, U_exact, cmap='viridis', alpha=0.9)
ax1.set_title('Exact $u = \\sin(\\pi x)\\sin(\\pi y)$')
ax1.set_xlabel('$x$'); ax1.set_ylabel('$y$'); ax1.set_zlabel('$u$')

# Spectral solution
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot_surface(X2, Y2, U_spec, cmap='plasma', alpha=0.9)
ax2.set_title(f'Spectral solution N={N_2d}')
ax2.set_xlabel('$x$'); ax2.set_ylabel('$y$'); ax2.set_zlabel('$u$')

# Error surface
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
err_surf = np.abs(U_spec - U_exact)
ax3.plot_surface(X2, Y2, err_surf, cmap='hot', alpha=0.9)
ax3.set_title(f'Pointwise Error (max={err_2d:.1e})')
ax3.set_xlabel('$x$'); ax3.set_ylabel('$y$'); ax3.set_zlabel('$|error|$')

plt.suptitle('2D Poisson: Spectral Collocation on Tensor Grid', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 2D Convergence Study
# =============================================================================

Nvals_2d = [4, 6, 8, 10, 12, 14, 16, 18, 20]
errs_2d = []
for N in Nvals_2d:
    U_s, xs, ys = spectral_poisson_2d(N, f_2d, g_2d)
    Xg, Yg = np.meshgrid(xs, ys, indexing='ij')
    errs_2d.append(np.max(np.abs(U_s - u_exact_2d(Xg, Yg))))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(Nvals_2d, errs_2d, 'b^-', markersize=7, label='2D Spectral Poisson')
ax.axhline(1e-15, color='k', ls=':', lw=1, label='Machine $\\epsilon$')
ax.set_title('2D Poisson Convergence: $\\nabla^2 u = f$ on $[-1,1]^2$')
ax.set_xlabel('$N$ per dimension')
ax.set_ylabel('$\\|u - u_{\\rm exact}\\|_\\infty$')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("2D spectral methods inherit the exponential convergence of 1D via tensor products.")

## 9. Time-Dependent PDEs — Method of Lines

### Heat Equation with Spectral Spatial Discretization

Consider the heat equation:
$$u_t = \nu \, u_{xx}, \quad x \in [-1,1], \quad t \in [0, T]$$
$$u(\pm 1, t) = 0, \quad u(x, 0) = u_0(x)$$

**Exact solution** (for $u_0(x) = \sin(k\pi x/2)$ satisfying BCs): 
$$u(x,t) = \sin(k\pi x/2)\, e^{-\nu (k\pi/2)^2 t}$$

### Method of Lines

Discretize **only** in space using the Chebyshev differentiation matrix, leaving time continuous:

$$\frac{d\mathbf{u}}{dt} = \nu D^2 \mathbf{u}$$

with $u_0 = u_N = 0$ enforced at every time step. This reduces the PDE to a system of ODEs:

$$\dot{\mathbf{u}}_{\rm int} = \nu (D^2)_{\rm int} \, \mathbf{u}_{\rm int}$$

We integrate in time with **4th-order Runge-Kutta (RK4)**:

$$\boxed{\begin{aligned}
k_1 &= h\,F(t_n,\mathbf{u}_n) \\
k_2 &= h\,F(t_n+h/2,\mathbf{u}_n+k_1/2) \\
k_3 &= h\,F(t_n+h/2,\mathbf{u}_n+k_2/2) \\
k_4 &= h\,F(t_n+h,\mathbf{u}_n+k_3) \\
\mathbf{u}_{n+1} &= \mathbf{u}_n + \tfrac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4)
\end{aligned}}$$

**Stability note**: The Chebyshev nodes cluster near the boundary, so the largest eigenvalue of $D^2$ grows as $O(N^4)$ — this severely restricts the time step for explicit methods. The stability limit is $\Delta t \lesssim 8/N^4$ for RK4.

In [ ]:
# =============================================================================
# RK4 Integrator (from scratch)
# =============================================================================

def rk4_step(F, t, u, dt):
    """Single step of the classical 4th-order Runge-Kutta method.

    Args:
        F (callable): RHS function F(t, u) -> du/dt, returns array of same shape as u.
        t (float): Current time.
        u (np.ndarray): Current state, shape (n,).
        dt (float): Time step size.

    Returns:
        np.ndarray: State at t + dt, shape (n,).
    """
    k1 = dt * F(t,          u)
    k2 = dt * F(t + dt/2,   u + k1/2)
    k3 = dt * F(t + dt/2,   u + k2/2)
    k4 = dt * F(t + dt,     u + k3)
    return u + (k1 + 2*k2 + 2*k3 + k4) / 6.0


print("rk4_step() defined.")

In [ ]:
# =============================================================================
# Heat Equation: Method of Lines with Spectral Spatial Discretization
# u_t = nu * u_xx,  u(pm1, t) = 0,  u(x,0) = sin(pi*x)
# Exact: u(x,t) = sin(pi*x) * exp(-nu*pi^2*t)
# =============================================================================

nu  = 1.0          # diffusivity
N_heat = 24        # Chebyshev degree
T_end = 0.5        # final time

D_heat, x_heat = cheb(N_heat)
D2_heat = D_heat @ D_heat

# Initial condition
u0 = np.sin(np.pi * x_heat)   # satisfies u(pm1)=0 since sin(pm pi)=0

# Interior indices (exclude endpoints)
int_idx = slice(1, N_heat)   # indices 1 .. N-1
D2_int  = D2_heat[int_idx, int_idx]   # (N-1) x (N-1) block (BCs=0, so edge columns vanish)

# But the full D2 has cross-terms with boundary nodes. Since u_boundary=0 always,
# we can simply extract the interior block and work with it.
# Actually, we need columns for boundary nodes too (they contribute 0):
# D2[int, :] @ u = D2[int, int] @ u_int + D2[int, bdy] @ u_bdy
# Since u_bdy = 0 always, D2_int = D2[int_idx, int_idx] is correct.

def heat_rhs(t, u_int):
    """RHS for heat equation (interior nodes only)."""
    return nu * D2_int @ u_int

# Time step: must satisfy RK4 stability for spectral Laplacian
# lambda_max ~ nu * max|eig(D2)|. For safety use dt = 8/(nu * N^4) * small factor
lam_max = nu * np.max(np.abs(np.linalg.eigvals(D2_heat)))
dt = 2.0 / lam_max   # RK4 stability requires dt * |lam_max| < ~2.79
Nsteps = int(np.ceil(T_end / dt))
dt = T_end / Nsteps
print(f"N={N_heat}, lambda_max={lam_max:.1f}, dt={dt:.2e}, Nsteps={Nsteps}")

# Time integration
u_int = u0[int_idx].copy()
t_curr = 0.0
snap_times = [0.0, 0.1, 0.2, 0.5]
snaps = {0.0: u0.copy()}

for step in range(Nsteps):
    u_int = rk4_step(heat_rhs, t_curr, u_int, dt)
    t_curr += dt
    for ts in snap_times[1:]:
        if abs(t_curr - ts) < dt/2 and ts not in snaps:
            u_full = np.zeros(N_heat + 1)
            u_full[int_idx] = u_int
            snaps[ts] = u_full.copy()

# Final state
u_full_final = np.zeros(N_heat + 1)
u_full_final[int_idx] = u_int
snaps[T_end] = u_full_final

# Exact solution
u_exact_heat = lambda x, t: np.sin(np.pi * x) * np.exp(-nu * np.pi**2 * t)

err_heat = np.max(np.abs(u_full_final - u_exact_heat(x_heat, T_end)))
print(f"Heat equation final error at T={T_end}: {err_heat:.2e}  [{'PASS' if err_heat < 1e-6 else 'FAIL'}]")

In [ ]:
# =============================================================================
# Visualize Heat Equation Solution
# =============================================================================

x_fine = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cmap_t = plt.cm.coolwarm
snap_keys = sorted(snaps.keys())
for i, ts in enumerate(snap_keys):
    col = cmap_t(i / (len(snap_keys) - 1))
    u_plot = snaps[ts]
    ax.plot(x_fine, u_exact_heat(x_fine, ts), '-', color=col, lw=2,
            label=f'Exact $t={ts:.1f}$', zorder=5)
    ax.plot(x_heat, u_plot, 'o', color=col, markersize=5, zorder=6)

ax.set_title('Heat Equation $u_t = u_{xx}$: Method of Lines + RK4')
ax.set_xlabel('$x$'); ax.set_ylabel('$u(x,t)$')
ax.legend(fontsize=9)

# Space-time heatmap
ax = axes[1]
t_vals = np.linspace(0, T_end, 100)
X_st, T_st = np.meshgrid(x_fine, t_vals)
U_st = u_exact_heat(X_st, T_st)
im = ax.contourf(X_st, T_st, U_st, levels=20, cmap='hot')
plt.colorbar(im, ax=ax)
ax.set_title('Space-Time Diagram: Heat Equation')
ax.set_xlabel('$x$'); ax.set_ylabel('$t$')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Heat Equation: Convergence in N (spatial) at fixed time T
# =============================================================================

T_conv = 0.2
Nvals_heat = [4, 6, 8, 10, 12, 14, 16, 18, 20]
errs_heat_N = []

for N in Nvals_heat:
    D_h, x_h = cheb(N)
    D2_h = D_h @ D_h
    idx_int = slice(1, N)
    D2_int_h = D2_h[idx_int, idx_int]

    u0_h = np.sin(np.pi * x_h)
    u_int_h = u0_h[idx_int].copy()

    lam_h = nu * np.max(np.abs(np.linalg.eigvals(D2_h)))
    dt_h  = 1.8 / lam_h
    Ns_h  = int(np.ceil(T_conv / dt_h))
    dt_h  = T_conv / Ns_h

    rhs_h = lambda t, u: nu * D2_int_h @ u

    t_h = 0.0
    for _ in range(Ns_h):
        u_int_h = rk4_step(rhs_h, t_h, u_int_h, dt_h)
        t_h += dt_h

    u_full_h = np.zeros(N + 1)
    u_full_h[idx_int] = u_int_h
    err_h = np.max(np.abs(u_full_h - u_exact_heat(x_h, T_conv)))
    errs_heat_N.append(err_h)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(Nvals_heat, errs_heat_N, 'mo-', markersize=7, label='Heat eq. spatial error')
ax.axhline(1e-15, color='k', ls=':', lw=1, label='Machine $\\epsilon$')
ax.set_title(f'Heat Equation: Spatial Convergence at $T={T_conv}$')
ax.set_xlabel('$N$ (Chebyshev degree)')
ax.set_ylabel('$\\|u_N(\\cdot, T) - u_{\\rm exact}(\\cdot, T)\\|_\\infty$')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Spectral spatial convergence is exponential; temporal error (RK4) is O(dt^4).")

## 10. Summary, Key Takeaways & References

### Summary Table

| Component | Description | Key Result |
|---|---|---|
| Chebyshev polynomials | $T_n(x)=\cos(n\arccos x)$, recurrence $T_{n+1}=2xT_n-T_{n-1}$ | Minimax optimal, orthogonal w.r.t. $w=(1-x^2)^{-1/2}$ |
| Gauss-Lobatto nodes | $x_j = \cos(j\pi/N)$, cluster at $\pm 1$ | Lebesgue constant $O(\log N)$, avoids Runge phenomenon |
| Differentiation matrix | $D_{jk} = \frac{c_j}{c_k}\frac{(-1)^{j+k}}{x_j-x_k}$, diagonal by negative sum | Dense $(N+1)^2$ matrix, machine-precision derivatives |
| 1D BVP collocation | Replace BC rows, solve $(D^2+PD+Q)u=f$ | Exponential convergence $\|e\|\sim\rho^{-N}$ |
| 2D Poisson | Kronecker product $L=I\otimes D^2+D^2\otimes I$ | Same exponential rate, $(N+1)^2$ DOF |
| Heat equation (MoL) | Spectral in $x$, RK4 in $t$ | Spatial: exponential; temporal: $O(\Delta t^4)$ |
| vs. Finite Differences | FD: $O(h^2)$; Spectral: $O(\rho^{-N})$ | Spectral needs $\sim$10–100× fewer DOF for same accuracy |

### Key Takeaways

1. **Exponential convergence** — For analytic solutions, spectral methods deliver machine-precision accuracy with surprisingly few degrees of freedom (typically $N \sim 15$–$30$).

2. **The `cheb(N)` function** — A single 10-line routine constructs the full differentiation matrix via the barycentric formula. Higher-order operators come for free by matrix multiplication: $D^2, D^3$, etc.

3. **Node choice matters critically** — Gauss-Lobatto nodes with their $\arcsin$-like distribution near boundaries are essential; equispaced nodes lead to catastrophic Runge instability.

4. **Collocation for BVPs** — Imposing BCs by replacing rows in the differentiation matrix system is straightforward and robust. Variable-coefficient problems require no special treatment.

5. **Kronecker products for 2D** — Tensor product construction extends 1D spectral methods to multidimensions without new formulas: $L_{2D} = I \otimes D^2 + D^2 \otimes I$.

6. **Method of lines** — Separating spatial and temporal discretizations allows use of any ODE integrator. The severe eigenvalue stiffness ($\lambda_{\max} \sim N^4$) of $D^2$ makes implicit time-stepping advantageous for large $N$.

7. **When to use spectral methods** — Best for smooth solutions on simple geometries. Discontinuities/shocks cause Gibbs oscillations; complex domains require domain decomposition or mapped coordinates.

### References

1. **Trefethen, L. N.** (2000). *Spectral Methods in MATLAB*. SIAM. (Primary reference — this notebook follows its structure closely.)
2. **Boyd, J. P.** (2001). *Chebyshev and Fourier Spectral Methods* (2nd ed.). Dover.
3. **Canuto, C., Hussaini, M. Y., Quarteroni, A., & Zang, T. A.** (2006). *Spectral Methods: Fundamentals in Single Domains*. Springer.
4. **Fornberg, B.** (1996). *A Practical Guide to Pseudospectral Methods*. Cambridge University Press.
5. **Shen, J., Tang, T., & Wang, L.-L.** (2011). *Spectral Methods: Algorithms, Analysis and Applications*. Springer.
6. **Mason, J. C., & Handscomb, D. C.** (2003). *Chebyshev Polynomials*. CRC Press.